In [0]:
def update_watermark(catalog, schema_name, table_name, last_modified):
    """
    Inserts or updates the watermark for a given table after a successful run.
    
    Only call this after a successful write. If last_modified is None, the watermark is not updated and a warning is logged.
    """
    columns_to_check = ['last_modified_datetime', 'TimeModified']
    watermark_table = f"{catalog}.{schema_name}.etl_watermark"
    full_table_name = f"{catalog}.{schema_name}.{table_name}"

    # Find the max modified timestamp from the target table
    try:
        table_columns = {col.name for col in spark.table(full_table_name).schema}

        matched_column = None
        for col in columns_to_check:
            if col in table_columns:
                matched_column = col
                break

        if matched_column is None:
            logger.warning(f"No modified column found in {full_table_name}. Checked: {columns_to_check}. Available: {sorted(table_columns)}. Watermark not updated.")
            return

        result = spark.sql(f"""SELECT MAX(`{matched_column}`) AS max_modified FROM {full_table_name}""").collect()

        if not result or result[0][0] is None:
            logger.warning(f"Table {full_table_name} is empty - watermark not updated")
            return

        last_modified = str(result[0][0])
        logger.info(f"Max modified for {table_name} ({matched_column}): {last_modified}")

    except Exception as e:
        logger.error(f"Error reading max modified from {full_table_name}: {e}")
        return

    # MERGE into the watermark table
    try:
        if not spark.catalog.tableExists(watermark_table):
            logger.error(f"Watermark table does not exist: {watermark_table} - cannot update")
            raise RuntimeError(f"Watermark table does not exist: {watermark_table}")

        spark.sql(f"""
            MERGE INTO {watermark_table} AS target
            USING (SELECT '{table_name}' AS table_name, CAST('{last_modified}' AS TIMESTAMP) AS last_modified, current_timestamp() AS updated_at) AS source
            ON target.table_name = source.table_name
            WHEN MATCHED THEN UPDATE SET
                target.last_modified = source.last_modified,
                target.updated_at = source.updated_at
            WHEN NOT MATCHED THEN INSERT (table_name, last_modified, updated_at)
                VALUES (source.table_name, source.last_modified, source.updated_at)
        """)

        logger.info(f"Watermark updated for {table_name}: {last_modified}")

    except Exception as e:
        logger.error(f"Failed to update watermark for {table_name}: {e}")
        raise